In [16]:
!pip install gradio transformers accelerate torch

In [17]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import tempfile

In [18]:
model_name = "ibm-granite/granite-3.3-2b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

In [19]:
class ColorPaletteGenerator:

    palettes = {
        "modern": ["#2563eb","#1e293b","#f1f5f9"],
        "ocean": ["#0ea5e9","#0284c7","#e0f2fe"],
        "forest": ["#16a34a","#166534","#dcfce7"],
        "sunset": ["#f97316","#ea580c","#fff7ed"],
        "luxury": ["#111827","#374151","#f9fafb"],
        "creative": ["#9333ea","#c084fc","#faf5ff"],
        "minimal": ["#6b7280","#374151","#f9fafb"],
        "vibrant": ["#ec4899","#db2777","#fdf2f8"]
    }

    def get_palette(self, theme):
        return self.palettes.get(theme,self.palettes["modern"])

In [20]:
class CSSGenerator:

    def __init__(self, palette):
        self.primary = palette[0]
        self.secondary = palette[1]
        self.bg = palette[2]

    def generate_css(self):

        return f"""
        body {{
        font-family: Arial;
        margin:0;
        background:{self.bg};
        }}

        .hero {{
        background:{self.primary};
        color:white;
        padding:80px;
        text-align:center;
        }}

        .section {{
        padding:40px;
        text-align:center;
        }}

        .card {{
        display:inline-block;
        margin:15px;
        padding:20px;
        border-radius:10px;
        background:white;
        box-shadow:0 3px 12px rgba(0,0,0,0.1);
        }}

        footer {{
        background:{self.secondary};
        color:white;
        text-align:center;
        padding:20px;
        }}
        """

In [21]:
class SectionTemplates:

    def hero_section(self,title,subtitle):

        return f"""
        <div class="hero">
        <h1>{title}</h1>
        <p>{subtitle}</p>
        <button>Get Started</button>
        </div>
        """

    def features_section(self):

        return """
        <div class="section">
        <h2>Features</h2>

        <div class="card">Fast Performance</div>
        <div class="card">Modern Design</div>
        <div class="card">Responsive Layout</div>

        </div>
        """

    def portfolio_section(self):

        return """
        <div class="section">
        <h2>Portfolio</h2>

        <div class="card">Project One</div>
        <div class="card">Project Two</div>
        <div class="card">Project Three</div>

        </div>
        """

    def testimonials_section(self):

        return """
        <div class="section">
        <h2>Testimonials</h2>

        <p>"Amazing work!"</p>
        <p>"Very professional"</p>

        </div>
        """

    def contact_section(self):

        return """
        <div class="section">
        <h2>Contact</h2>

        <p>Email: contact@example.com</p>
        <p>Phone: +91 9999999999</p>

        </div>
        """

In [26]:
def generate_website(title,subtitle,theme,sections):

    palette = ColorPaletteGenerator().get_palette(theme)

    css = CSSGenerator(palette).generate_css()

    templates = SectionTemplates()

    html = f"""
    <html>
    <head>
    <title>{title}</title>

    <style>
    {css}
    </style>

    </head>

    <body>
    """

    html += templates.hero_section(title,subtitle)

    if "Features" in sections:
        html += templates.features_section()

    if "Portfolio" in sections:
        html += templates.portfolio_section()

    if "Testimonials" in sections:
        html += templates.testimonials_section()

    if "Contact" in sections:
        html += templates.contact_section()

    html += """
    <footer>

    Generated using HTML Quick Styler

    </footer>

    </body>
    </html>
    """

    return html,html

In [25]:
import tempfile

def download_html(title,subtitle,theme,sections):

    html, _ = generate_website(title,subtitle,theme,sections)

    file = tempfile.NamedTemporaryFile(delete=False,suffix=".html")

    with open(file.name,"w") as f:
        f.write(html)

    return file.name

In [27]:
with gr.Blocks() as demo:

    gr.Markdown("# HTML Quick Styler - AI Webpage Generator")

    with gr.Row():

        with gr.Column():

            title = gr.Textbox(label="Website Title")
            subtitle = gr.Textbox(label="Subtitle / Description")

            theme = gr.Dropdown(
                ["modern","ocean","forest","sunset","luxury","creative","minimal","vibrant"],
                label="Color Theme"
            )

            sections = gr.CheckboxGroup(
                ["Features","Portfolio","Testimonials","Contact"],
                label="Sections"
            )

            generate_btn = gr.Button("Generate Website")

        with gr.Column():

            html_code = gr.Code(
                label="Generated HTML Code",
                language="html"
            )

    preview = gr.HTML()

    download_btn = gr.File(label="Download Generated HTML")

    generate_btn.click(
        fn=generate_website,
        inputs=[title, subtitle, theme, sections],
        outputs=[html_code, preview]
    )

    generate_btn.click(
        fn=download_html,
        inputs=[title, subtitle, theme, sections],
        outputs=download_btn
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://741afe6509d2657eaf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
